In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset


In [2]:
url = 'data/'


In [4]:
df = pd.read_parquet(url + 'AAPL_1minute.parquet')

df.head()

,date,open,high,low,close,volume
0,2020-05-11 08:00:00,77.7850,77.7850,77.7850,77.7850,1496
1,2020-05-11 08:06:00,77.8375,77.9125,77.8375,77.9125,2224
2,2020-05-11 08:08:00,77.8375,77.8425,77.7850,77.7875,16836
3,2020-05-11 08:12:00,77.7850,77.7850,77.7850,77.7850,4000
4,2020-05-11 08:13:00,77.7775,77.7775,77.7775,77.7775,760


In [5]:
df.describe()

,date,open,high,low,close,volume
count,920803,920803.000000,920803.000000,920803.000000,920803.000000,9.208030e+05
mean,2022-08-15 16:36:07.017136,158.997272,159.048927,158.944687,158.997166,9.475101e+04
min,2020-05-11 08:00:00,75.012600,75.095000,74.937500,75.017500,1.000000e+02
25%,2021-06-17 13:53:30,134.063600,134.110000,134.020000,134.065000,1.736000e+03
50%,2022-08-03 11:12:00,154.980000,155.030000,154.930000,154.980000,3.205700e+04
75%,2023-10-04 18:26:30,178.840000,178.890000,178.795000,178.840000,1.260675e+05
max,2025-01-01 00:58:00,259.910000,260.100000,259.710000,259.929900,1.940765e+07
std,NaN,35.263817,35.268660,35.259077,35.264066,1.946441e+05


Preparação das Janelas de Séries Temporais

In [6]:
# Definindo o tamanho da janela de observação para as séries temporais
janela = 30  # Número de passos de tempo usados como entrada para o modelo

# Lista de colunas de features e o alvo (target) para predição
features = ['open', 'high', 'low', 'close', 'volume']
target = 'close'

# Importando bibliotecas necessárias
import numpy as np
from sklearn.model_selection import train_test_split

# Função para criar janelas de séries temporais a partir do DataFrame
# Cada janela contém 'janela' linhas das features e o valor alvo do próximo passo
def criar_janelas(df, janela, features, target):
    X, y = [], []
    for i in range(len(df) - janela):
        # Seleciona uma janela de dados das features
        X.append(df.iloc[i:i+janela][features].values)
        # Seleciona o valor alvo correspondente ao final da janela
        y.append(df.iloc[i+janela][target])
    return np.array(X), np.array(y)

# Criação das janelas de entrada (X) e saída (y) para o modelo
# Certifique-se de que 'df' já está carregado com os dados antes de executar esta célula
X, y = criar_janelas(df, janela, features, target)

# Separação dos dados em treino e teste (80% treino, 20% teste, sem embaralhar)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

NameError: name 'criar_janelas' is not defined

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)